# Example for [passagemath-cmr](https://pypi.org/project/passagemath-cmr/)

This notebook illustrates Seymour's decomposition of totally unimodular matrices and regular matroids provided by [passagemath-cmr](https://pypi.org/project/passagemath-cmr/) – one of the modularized pip-installable packages of the Sage library provided by the [passagemath project](https://github.com/passagemath).

If packages are not already loaded through the marimo sandbox, use the "Packages" tab on the left to uv-install `passagemath-cmr[test] passagemath-polyhedra[flint] passagemath-nauty passagemath-plot` for the functionality demonstrated in this marimo notebook.

In [ ]:
import marimo as mo
import passagemath_pari
import passagemath_polyhedra, passagemath_flint, passagemath_graphs, passagemath_nauty, passagemath_repl
from passagemath_cmr import matrix, unicode_art
from passagemath_graphs import matroids, Matroid, Graph, DiGraph, QQ, ZZ, graphs, digraphs
import sage.typeset.character_art
sage.typeset.character_art.MAX_WIDTH = 200

## 3.1 Matrices

The pip-installable package `passagemath-cmr` extends standard matrix types with specialized methods for Seymour's decomposition and recognition of totally unimodular (TU) matrices.

In [ ]:
A = matrix([[1, 0], [-1, -1], [0, 1]], column_keys=['a', 'b'], row_keys=range(3)); A

In [ ]:
A._unicode_art_matrix()

In [ ]:
result, certificate = A.is_totally_unimodular(certificate=True); result, certificate

In [ ]:
certificate.graph()

All of these methods are made available by delegating to a specialized matrix element class `Matrix_cmr_chr_sparse`, in which matrices are backed by the CMR library.

## 3.2 Module morphisms

The package `passagemath-modules` provides linear algebra facilities in a style favored in algebraic combinatorics. Users can define vector spaces and free modules with distinguished bases whose elements are indexed by arbitrary objects. Linear maps (module morphisms) between such vector spaces or modules are represented by matrices whose rows and columns are indexed by the basis indices.

In [ ]:
A2 = matrix([[-1,  0,  0,  0,  1, -1,  0],
             [ 1,  0,  0,  1, -1,  1,  0],
             [ 0, -1,  0, -1,  1, -1,  0],
             [ 0,  1,  0,  0,  0,  0,  1],
             [ 0,  0,  1, -1,  1,  0,  1],
             [ 0,  0, -1,  1, -1,  0,  0]],
            column_keys=['a', 'b', 'c', 'd', 'e', 'f', 'g'],
            row_keys=range(6))

In [ ]:
A2._unicode_art_matrix()

In [ ]:
A2_result, A2_certificate = A2.is_totally_unimodular(certificate=True); A2_result, A2_certificate

In [ ]:
A2_certificate.graph().incidence_matrix(vertices=True,edges=True)._unicode_art_matrix()

## 3.3 Graphs

Graph structures have a direct link to total unimodularity. The vertex-edge incidence matrix of any directed graph is totally unimodular.

In [ ]:
G_directed = DiGraph([(0, 1), (1, 2), (2, 0)])
m_directed = G_directed.incidence_matrix(oriented=True)
res_directed, cert_directed = m_directed.is_totally_unimodular(certificate=True)

In [ ]:
res_directed, cert_directed

For undirected graphs, the incidence matrix is totally unimodular if and only if the graph is bipartite.

In [ ]:
c4 = Graph([(0, 1), (1, 2), (2, 3), (3, 0)])
c5 = Graph([(0, 1), (1, 2), (2, 3), (3, 4), (4, 0)])
res_c4 = c4.incidence_matrix().is_totally_unimodular()
res_c5, cert_c5 = c5.incidence_matrix().is_totally_unimodular(certificate=True)

In [ ]:
print("C4 is TU:", res_c4)
print("C5 is TU:", res_c5)
print("C5 certificate violating root node:", cert_c5[0])

### 3.3.1 Odd Cycle Packing Number

We can check whether a connected undirected graph has odd cycle packing number $\mathrm{ocp}(G) \le 1$. It is done by first checking if the incidence matrix is totally unimodular. If so, it is a bipartite graph with no odd cycle ($\mathrm{ocp}(G) = 0$). If not, we can find a basis $B$, which contains at least one odd cycle (otherwise it cannot be a basis). Then by left multiplying the inverse of the basis and checking if the obtained matrix is totally unimodular, we can claim that the graph has odd cycle packing number $\log_2(|\det(B)|)$ (which is 1) if the obtained matrix is totally unimodular.

In the following implementation, we use the `rref` function to compute the new matrix after left multiplying the inverse of the basis for efficiency.

In [ ]:
gt = graphs.GrotzschGraph()
At = gt.incidence_matrix(vertices=True, edges=True).matrix()
res_gt_tu, cert_gt_tu = At.is_totally_unimodular(certificate=True)
res_gt_rref_tu, cert_gt_rref_tu = At.rref().is_totally_unimodular(certificate=True)

In [ ]:
print("Grötzsch Graph incidence matrix is TU:", res_gt_tu)
print("Grötzsch Graph certificate:", cert_gt_tu)
print("Grötzsch Graph rref matrix is TU:", res_gt_rref_tu)
print("Grötzsch Graph rref certificate:", cert_gt_rref_tu)

In [ ]:
A_K6 = graphs.CompleteGraph(6).incidence_matrix()
res_K6_tu, cert_K6_tu = A_K6.is_totally_unimodular(certificate=True)
res_K6_rref_tu, cert_K6_rref_tu = A_K6.rref().is_totally_unimodular(certificate=True)

In [ ]:
print("K6 incidence matrix is TU:", res_K6_tu)
print("K6 certificate:", cert_K6_tu)
print("K6 rref matrix is TU:", res_K6_rref_tu)
print("K6 rref certificate:", cert_K6_rref_tu)

### 3.3.2 Recognition algorithm of network matrices

The `passagemath-cmr` package also provides the recognition algorithm of network matrices. It can return the directed graph certificate of a network matrix, which can be used to reconstruct the original matrix.

In [ ]:
A2_g = matrix([[-1,  0,  0,  0,  1, -1,  0],
             [ 1,  0,  0,  1, -1,  1,  0],
             [ 0, -1,  0, -1,  1, -1,  0],
             [ 0,  1,  0,  0,  0,  0,  1],
             [ 0,  0,  1, -1,  1,  0,  1],
             [ 0,  0, -1,  1, -1,  0,  0]],
            column_keys=['a', 'b', 'c', 'd', 'e', 'f', 'g'],
            row_keys=range(6))

In [ ]:
A2_g_result, A2_g_certificate = A2_g.is_totally_unimodular(certificate=True); A2_g_result, A2_g_certificate

In [ ]:
G_g = A2_g_certificate.graph()

In [ ]:
M_g = G_g.incidence_matrix(vertices=True, edges=True)

In [ ]:
M_g._unicode_art_matrix()

In [ ]:
row_keys_g, forest_order_g = zip(*A2_g_certificate.forest_edges().items())
column_keys_g, coforest_order_g = zip(*A2_g_certificate.coforest_edges().items())
row_order_g = G_g.vertices()[:-1]
AA_g = M_g.matrix(row_order=row_order_g, column_order=forest_order_g).inverse() * M_g.matrix(row_order=row_order_g, column_order=coforest_order_g)
print(AA_g)

In [ ]:
matrix(AA_g, base_ring=ZZ, row_keys=row_keys_g, column_keys=column_keys_g) == A2_g

Here are some more examples of the two excluded minors $K_5$, $K_{3,3}$ for conetwork/network matrices.

In [ ]:
K5 = digraphs.Complete(5)
M_K5 = K5.incidence_matrix()

In [ ]:
M_K5.is_conetwork_matrix()

In [ ]:
M_K5.is_network_matrix()

In [ ]:
K33_undirect = graphs.CompleteBipartiteGraph(3, 3)
K33 = K33_undirect.orient(lambda e: e if e[0] < e[1] else (e[1], e[0], e[2]))
M_K33 = K33.incidence_matrix(vertices=True, edges=True)

In [ ]:
M_K33.is_conetwork_matrix()

In [ ]:
M_K33.is_network_matrix()

In [ ]:
res_K33, cert_K33 = M_K33.is_totally_unimodular(certificate=True)

In [ ]:
cert_K33

## 3.4 Matroids

The packages `passagemath-graphs` and `passagemath-modules` provide facilities for matroid theory. A comprehensive catalog of known matroids is a good starting point for investigations. We can inspect the regular matroid $R_{10}$ and perform Seymour's decomposition.

In [ ]:
R10 = matroids.catalog.R10()

In [ ]:
sorted(R10.groundset())

In [ ]:
R10_rr = R10.representation(reduced=True, order=True)
# Reference A from cell 4
R10_rr, A

In [ ]:
R10_rr._unicode_art_matrix()

In [ ]:
R10_tu, R10_certificate = R10_rr.is_totally_unimodular(certificate=True); R10_certificate

In [ ]:
R10_certificate.morphism()

In [ ]:
R10_certificate.morphism()._unicode_art_matrix()

In [ ]:
R10D = R10.dual(); R10D

In [ ]:
R10D_rr = R10D.representation(reduced=True, order=True); R10D_rr

In [ ]:
R10D_rr._unicode_art_matrix()

In [ ]:
R10D.is_isomorphic(R10)

In [ ]:
R10D_tu, R10D_certificate = R10D_rr.is_totally_unimodular(certificate=True); R10D_certificate

In [ ]:
R10_1_R10 = R10.direct_sum(R10)
R10_1_R10_reg = Matroid(R10_1_R10, regular=True)
R10_1_R10_reg.representation()

In [ ]:
R10_1_R10_rr = R10_1_R10_reg.representation(reduced=True, order=True); R10_1_R10_rr

In [ ]:
R10_1_R10_tu, R10_1_R10_certificate = R10_1_R10_rr.is_totally_unimodular(certificate=True); R10_1_R10_certificate

In [ ]:
unicode_art(R10_1_R10_certificate.as_ordered_tree())

In [ ]:
R10_1_R10_rr._unicode_art_matrix()

In [ ]:
R10_1_R10_certificate.block_matrix_form()

### 3.4.1 Non-Regular Matroids (AG23minus)

Not all matroids are regular. The `AG23minus` matroid is representable over GF(3) (ternary) but is not binary, meaning it cannot be represented by a totally unimodular matrix.

In [ ]:
AG23minus = matroids.catalog.AG23minus()

In [ ]:
AG23minus.is_regular()

In [ ]:
AG23minus.is_binary(), AG23minus.is_ternary()

In [ ]:
AG23minus_ternary = AG23minus.ternary_matroid()

In [ ]:
AG23minus_rr = AG23minus_ternary.representation(reduced=True, order=True)

In [ ]:
AG23minus_rr._unicode_art_matrix()

In [ ]:
AG23minus_tu, AG23minus_certificate = AG23minus_rr.is_totally_unimodular(certificate=True); AG23minus_certificate

## 3.5 Polyhedra and linear programming

The Sage library provides a simple modeling facility for mixed-integer linear programs with access to various numerical solvers as backends, as well as facilities for convex polyhedra. In the passagemath system, this functionality is available in the pip-installable package **passagemath-polyhedra**.

In [ ]:
from sage.numerical.mip import MixedIntegerLinearProgram

### 3.5.1 Network flow

As an illustrating example, we set up a min-cost flow problem.

In [ ]:
GP = graphs.PetersenGraph(); GP

In [ ]:
DP = next(GP.acyclic_orientations()); DP

In [ ]:
DPA = DP.incidence_matrix(vertices=True, edges=True); DPA

In [ ]:
DPA._unicode_art_matrix()

To set up a feasible min-cost flow problem, we pick a flow imbalance from the range of the matrix (image of this linear map).

In [ ]:
DPA.image()

In [ ]:
import random
dom = DPA.domain()
random_flow = dom.sum(random.randint(0, 5) * dom.monomial(e) for e in dom.basis().keys())
imbalance = DPA(random_flow); imbalance

In [ ]:
from sage.modules.free_module_element import vector
imbalance_vector = vector(QQ, [imbalance[v] for v in sorted(imbalance.parent().basis().keys())])
Mincostflow = MixedIntegerLinearProgram(solver='GLPK', base_ring=QQ)
flow = Mincostflow.new_variable(real=True, nonnegative=True, name="x"); flow
Mincostflow.add_constraint(DPA.matrix() * flow, min=imbalance_vector, max=imbalance_vector)

In [ ]:
Mincostflow.show()

In [ ]:
DPP = Mincostflow.polyhedron(base_ring=QQ)
print(DPP)

In [ ]:
DPP.vertices_matrix()

In [ ]:
DPP.rays_list()

### 3.5.2 Stable sets

For perfect graphs, the fractional stable set polyhedron (QSTAB) is equal to the stable set polyhedron (STAB), which is integral. Although QSTAB is integral for perfect graphs, the constraint matrix (clique-vertex incidence matrix) is not necessarily totally unimodular. When the matrix is non-TU, we can use the decomposition certificate to investigate the failure.

In [ ]:
def clique_vertex_incidence_matrix(G):
    cliques = [tuple(sorted(Q)) for Q in G.cliques_maximal()]
    vertices = list(G.vertices(sort=True))
    data = [[1 if v in clique else 0 for v in vertices] for clique in cliques]
    return matrix(data, column_keys=vertices, row_keys=cliques)

W6 = graphs.WheelGraph(6)
W6_clique_vertex_incidence_matrix = clique_vertex_incidence_matrix(W6)

W6_stab_mip = MixedIntegerLinearProgram(solver='GLPK', base_ring=QQ)
x = W6_stab_mip.new_variable(real=True, nonnegative=True, name="x")
for v in W6.vertices():
    W6_stab_mip.set_max(x[v], 1)
for clique in W6.cliques_maximal():
    W6_stab_mip.add_constraint(sum(x[v] for v in clique) <= 1)

In [ ]:
W6_clique_vertex_incidence_matrix._unicode_art_matrix()

In [ ]:
W6_qstab = W6_stab_mip.polyhedron(base_ring=QQ)
print(W6_qstab)

In [ ]:
W6_qstab.vertices_matrix()

The stable set polyhedron of $W_6$ contains fractional vertices (non-integral), because $W_6$ contains a $C_5$ cycle and is thus not perfect.

In [ ]:
W6_clique_vertex_incidence_matrix.is_totally_unimodular()

#### Interactive Wheel Graph Explorer

Let's check the total unimodularity of clique-vertex incidence matrices for the family of Wheel graphs $W_n$.

You can use the slider below to dynamically adjust the size of the Wheel Graph $W_n$ and inspect the total unimodularity of its clique-vertex incidence matrix.

In [ ]:
n_slider = mo.ui.slider(start=3, stop=20, step=1, value=6, label="Wheel Graph size (n)")
n_slider

In [ ]:
W_interactive = graphs.WheelGraph(n_slider.value)
W_interactive_tu, W_interactive_cert = clique_vertex_incidence_matrix(W_interactive).is_totally_unimodular(certificate=True)
W_interactive.plot()

In [ ]:
print(f"W_{n_slider.value} is totally unimodular: {W_interactive_tu}")
if not W_interactive_tu:
    print("Violation certificate root node:", W_interactive_cert[0])

#### Search for small non-unimodular examples
We can search for small non-unimodular graphs on 5 and 6 vertices.

In [ ]:
not_unimodular_on_5 = [G for G in graphs(5) 
                       if not clique_vertex_incidence_matrix(G).is_totally_unimodular()]
not_unimodular_on_5

In [ ]:
not_unimodular_on_5[0].is_cycle()

In [ ]:
not_unimodular_on_6 = [G for G in graphs(6) 
                       if G.is_connected() 
                       and not clique_vertex_incidence_matrix(G).is_totally_unimodular()]
not_unimodular_on_6

In [ ]:
not_unimodular_on_6[0].is_perfect()

In [ ]:
perfect_but_not_unimodular_on_6 = [G for G in graphs(6) 
                                   if G.is_connected() and G.is_perfect() 
                                   and not clique_vertex_incidence_matrix(G).is_totally_unimodular()]
perfect_but_not_unimodular_on_6

In [ ]:
for g in perfect_but_not_unimodular_on_6: 
    print(clique_vertex_incidence_matrix(g)._unicode_art_matrix())

In [ ]:
[
    unicode_art(
        clique_vertex_incidence_matrix(G)
        .is_totally_unimodular(certificate=True)[1][0]
        .as_ordered_tree()
    )
    for G in perfect_but_not_unimodular_on_6
]

In [ ]:
len(list(graphs(9)))  # OEIS A000088

In [ ]:
perfect_but_not_unimodular_on_7 = [G for G in graphs(7) 
                                   if G.is_connected() and G.is_perfect() 
                                   and not clique_vertex_incidence_matrix(G).is_totally_unimodular()]
perfect_but_not_unimodular_on_7

In [ ]:
from sage.matrix.seymour_decomposition import SeriesParallelReductionNode

In [ ]:
def interesting_stuff():
    for G in perfect_but_not_unimodular_on_7:
        result, certificate = clique_vertex_incidence_matrix(G).is_totally_unimodular(certificate=True, stop_when_nonTU=False)
        if not isinstance(certificate[0], SeriesParallelReductionNode):
            yield G, certificate[0].as_ordered_tree() 
for G345678, tree in interesting_stuff():
    print(clique_vertex_incidence_matrix(G345678)._unicode_art_matrix())
    print(tree)

## 3.6 Detailed example for Seymour's decomposition

We can analyze a larger matrix `MM` of size 16x16 to get one possible Seymour decomposition tree.

In [ ]:
MM = matrix([[ 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
             [ 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
             [ 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, -1, 0, 0, -1, -1],
             [ 0, -1, 0, -1, 1, 0, 0, 0, 0, 0, 1, -1, 0, 0, -1, -1],
             [ 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
             [ 0, -1, 0, -1, 0, 0, 0, 1, -1, -1, 1, -1, 0, 0, -1, -1],
             [ 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0],
             [ 0, 1, 0, 1, 0, 0, 1, 0, 1, 1, -1, 1, 0, 0, 1, 1],
             [ 0, 0, 0, 0, 0, -1, 0, -1, 1, 1, 0, 0, 0, 0, 0, 0],
             [ 0, -1, 0, -1, 0, 0, 0, 0, 0, -1, 1, -1, 0, 0, -1, -1],
             [ 0, 0, 0, 0, 0, -1, 0, -1, 0, 1, 0, 0, 0, 0, 0, 0],
             [ 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, -1, 1, -1, 0, 0, 0],
             [ 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, -1, 1, -1, 0, 0],
             [ 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, -1, 1, -1, -1],
             [ 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, -1, 0, 0, -1, 1, 1],
             [ 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]])

In [ ]:
MM_result, MM_certificate = MM.is_totally_unimodular(certificate=True); MM_result, MM_certificate

In [ ]:
unicode_art(MM_certificate.as_ordered_tree())

The certificate `MM_certificate` is a `SeriesParallelReductionNode`. Such a node indicates that the input matrix arises from a smaller matrix $M'$ (called the **core**) by successively adding zero/unit rows/columns, or duplicates/scalings of existing rows/columns.

We can retrieve this core matrix using the `.core()` method, which is defined specifically for `SeriesParallelReductionNode`.

In [ ]:
MM1 = MM_certificate.core()

In [ ]:
MM1.dimensions()

In [ ]:
core_row_keys, core_column_keys = MM_certificate.child_keys(); core_row_keys, core_column_keys

In [ ]:
MM_certificate.child_nodes()[0].child_nodes()